In [55]:
import numpy as np
import pandas as pd

# Functions

In [56]:
def lin_function(x, a=False):
    return (0.5 if a else 0.3) * x

def lin_bias(x):
    return x

def quad_function(x, a=False):
    return -(0.4 if a else 0.45) * x**2 + 10

def quad_bias(x):
    return - x**2

def cubic_function(x, a=False):
    return 3*x + x**3 * (0.2 if a else 0.02)

def cubic_bias(x):
    return x**3

def sin_function(x, a=False):
    return np.sin(x + (-2 if a else 0) ) * 5

def sin_bias(x):
    return np.sin(x) * 2

# Generate noise data

In [57]:
def generate_noise_data(function, bias_function, n_samples, x_true_range, noise_std=1.2, seed=1, a_prob=0.5):
    np.random.seed(seed)

    X_true = np.random.uniform(*x_true_range, n_samples)
    x_noise = np.random.normal(0, noise_std, n_samples)

    X_obs = X_true + x_noise
    
    # Generate categorical feature 'a' (0 or 1)
    a_categorical = np.random.binomial(1, a_prob, n_samples)
    
    # Apply function with categorical feature
    y = np.array([function(x, bool(a)) for x, a in zip(X_true, a_categorical)])

    return pd.DataFrame({
        "X_obs": X_obs, 
        "X_true": X_true, 
        "x_noise": x_noise,
        "x_bias": bias_function(X_true),
        "a": a_categorical,
        "y": y
    })

In [58]:
def generate_multi_environment_data(function, bias_function, n_samples, train_range, test_range_1, test_range_2, 
                                   val_range_1, val_range_2, n_val, noise_std=1.2, n_environments=10, seed_base=1, a_prob=0.5):
    """
    Generiere Daten mit mehreren Environments durch Aufteilung des Train-Ranges.
    Optional wird ein Validierungs-Split erstellt, wenn `val_range` angegeben ist.
    `val_range` kann als (start, end) oder (start, end, n_samples) angegeben werden.
    """
    # Aufteilen des train_range in n_environments gleichmäßige Bereiche
    train_start, train_end = train_range
    range_width = (train_end - train_start) / n_environments
    env_ranges = [(train_start + i * range_width, train_start + (i + 1) * range_width) for i in range(n_environments)]
    
    all_data = []
    n_samples_per_env = n_samples // n_environments
    
    # Generiere Daten für jedes Environment
    for domain_id, env_range in enumerate(env_ranges):
        env_data = generate_noise_data(function, bias_function, n_samples_per_env, env_range, 
                                     noise_std, seed=seed_base + domain_id, a_prob=a_prob)
        env_data["split"] = "train"
        env_data["domain_id"] = domain_id
        env_data["env_range_start"] = env_range[0]
        env_data["env_range_end"] = env_range[1]
        all_data.append(env_data)
    
    # Test-Daten
    test_data_1 = generate_noise_data(function, bias_function, test_range_1[2] if len(test_range_1) > 2 else 500, 
                                     test_range_1[:2], noise_std, seed=seed_base + n_environments, a_prob=a_prob)
    test_data_1["split"] = "test"
    test_data_1["domain_id"] = -1  # -1 für Test-Daten
    test_data_1["env_range_start"] = test_range_1[0]
    test_data_1["env_range_end"] = test_range_1[1]
    all_data.append(test_data_1)
    
    test_data_2 = generate_noise_data(function, bias_function, test_range_2[2] if len(test_range_2) > 2 else 500, 
                                     test_range_2[:2], noise_std, seed=seed_base + n_environments + 1, a_prob=a_prob)
    test_data_2["split"] = "test"
    test_data_2["domain_id"] = -1  # -1 für Test-Daten
    test_data_2["env_range_start"] = test_range_2[0]
    test_data_2["env_range_end"] = test_range_2[1]
    all_data.append(test_data_2)

    val_n = val_range_1[2] if len(val_range_1) > 2 else (n_val if n_val is not None else 500)
    val_data = generate_noise_data(function, bias_function, val_n, val_range_1[:2], 
                                noise_std, seed=seed_base + n_environments + 2, a_prob=a_prob)
    val_data["split"] = "val"
    val_data["domain_id"] = -1
    val_data["env_range_start"] = val_range_1[0]
    val_data["env_range_end"] = val_range_1[1]
    all_data.append(val_data)

    val_n = val_range_2[2] if len(val_range_2) > 2 else (n_val if n_val is not None else 500)
    val_data = generate_noise_data(function, bias_function, val_n, val_range_2[:2], 
                                noise_std, seed=seed_base + n_environments + 2, a_prob=a_prob)
    val_data["split"] = "val"
    val_data["domain_id"] = -1
    val_data["env_range_start"] = val_range_2[0]
    val_data["env_range_end"] = val_range_2[1]
    all_data.append(val_data)
    
    return pd.concat(all_data, ignore_index=True)

In [59]:
train_range = (-4, 4)
test_range_1 = (-6.5, -5.5)
test_range_2 = (5.5, 6.5)
val_range_1 = (-5.4, -4.1)
val_range_2 = (4.1, 5.4)

n_train = 1500
n_test = 500
n_val = 500

noise_std = 1.2
y_noise_std = 0.4

## Quad data

In [60]:
data = generate_multi_environment_data(
    function=quad_function,
    bias_function=quad_bias,
    n_samples=n_train,
    train_range=train_range,
    test_range_1=test_range_1,
    test_range_2=test_range_2,
    val_range_1=val_range_1,
    val_range_2=val_range_2,
    n_val=n_val,
    noise_std=noise_std,
    n_environments=10,
    seed_base=1
)

data.to_csv("../data/quad.csv", index=False)

data.head()

,X_obs,X_true,x_noise,x_bias,a,y,split,domain_id,env_range_start,env_range_end
0,-5.314123,-3.666382,-1.647741,-13.442360,0,3.950938,train,0,-4.0,-3.2
1,-3.045549,-3.423740,0.378191,-11.721998,1,5.311201,train,0,-4.0,-3.2
2,-2.984516,-3.999909,1.015393,-15.999268,1,3.600293,train,0,-4.0,-3.2
3,-4.789553,-3.758134,-1.031419,-14.123571,0,3.644393,train,0,-4.0,-3.2
4,-3.461940,-3.882595,0.420655,-15.074546,0,3.216454,train,0,-4.0,-3.2


## Cubic data

In [61]:
data = generate_multi_environment_data(
    function=cubic_function,
    bias_function=cubic_bias,
    n_samples=n_train,
    train_range=train_range,
    test_range_1=test_range_1,
    test_range_2=test_range_2,
    val_range_1=val_range_1,
    val_range_2=val_range_2,
    n_val=n_val,
    noise_std=noise_std,
    n_environments=10,
    seed_base=1
)

data.to_csv("../data/cubic.csv", index=False)

data.head()

,X_obs,X_true,x_noise,x_bias,a,y,split,domain_id,env_range_start,env_range_end
0,-5.314123,-3.666382,-1.647741,-49.284832,0,-11.984844,train,0,-4.0,-3.2
1,-3.045549,-3.423740,0.378191,-40.133079,1,-18.297837,train,0,-4.0,-3.2
2,-2.984516,-3.999909,1.015393,-63.995608,1,-24.798847,train,0,-4.0,-3.2
3,-4.789553,-3.758134,-1.031419,-53.078271,0,-12.335967,train,0,-4.0,-3.2
4,-3.461940,-3.882595,0.420655,-58.528362,0,-12.818353,train,0,-4.0,-3.2


## Sin function

In [62]:
data = generate_multi_environment_data(
    function=sin_function,
    bias_function=sin_bias,
    n_samples=n_train,
    train_range=train_range,
    test_range_1=test_range_1,
    test_range_2=test_range_2,
    val_range_1=val_range_1,
    val_range_2=val_range_2,
    n_val=n_val,
    noise_std=noise_std * 0.5,
    n_environments=10,
    seed_base=1
)

data.to_csv("../data/sin.csv", index=False)

data.head()

,X_obs,X_true,x_noise,x_bias,a,y,split,domain_id,env_range_start,env_range_end
0,-4.490253,-3.666382,-0.823870,1.002062,0,2.505155,train,0,-4.0,-3.2
1,-3.234645,-3.423740,0.189096,0.556838,1,3.787401,train,0,-4.0,-3.2
2,-3.492212,-3.999909,0.507696,1.513485,1,1.397517,train,0,-4.0,-3.2
3,-4.273844,-3.758134,-0.515710,1.156433,0,2.891084,train,0,-4.0,-3.2
4,-3.672268,-3.882595,0.210328,1.350056,0,3.375140,train,0,-4.0,-3.2


# lin function

In [63]:
data = generate_multi_environment_data(
    function=lin_function,
    bias_function=lin_bias,
    n_samples=n_train,
    train_range=train_range,
    test_range_1=test_range_1,
    test_range_2=test_range_2,
    val_range_1=val_range_1,
    val_range_2=val_range_2,
    n_val=n_val,
    noise_std=noise_std * 0.5,
    n_environments=10,
    seed_base=1
)

data.to_csv("../data/lin.csv", index=False)

data.head()

,X_obs,X_true,x_noise,x_bias,a,y,split,domain_id,env_range_start,env_range_end
0,-4.490253,-3.666382,-0.823870,-3.666382,0,-1.099915,train,0,-4.0,-3.2
1,-3.234645,-3.423740,0.189096,-3.423740,1,-1.711870,train,0,-4.0,-3.2
2,-3.492212,-3.999909,0.507696,-3.999909,1,-1.999954,train,0,-4.0,-3.2
3,-4.273844,-3.758134,-0.515710,-3.758134,0,-1.127440,train,0,-4.0,-3.2
4,-3.672268,-3.882595,0.210328,-3.882595,0,-1.164779,train,0,-4.0,-3.2
